# 📊 Model Comparison

## Objective

Compare multiple regression models using the same dataset,
preprocessing pipeline, and evaluation metrics.

Models

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
df = pd.read_csv("../data/processed/house_prices_refined.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (13297, 5)


,bhk,propertytype,location,sqft,totalprice
0,3,Flat,Ahmedabad,1346,15700000
1,4,Flat,Ahmedabad,1872,17500000
2,4,Flat,Ahmedabad,1650,20200000
3,5,Flat,Ahmedabad,10201,86700000
4,3,Flat,Ahmedabad,968,10400000


In [16]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

target_column = "totalprice"

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Features:", feature_columns)
print("Target:", target_column)

Features: ['bhk', 'propertytype', 'location', 'sqft']
Target: totalprice


In [17]:
groups = (
    X.astype(str)
     .agg("||".join, axis=1)
)

print("Total rows:", len(df))
print("Unique feature groups:", groups.nunique())

Total rows: 13297
Unique feature groups: 9359


In [18]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (10556, 4)
X_test shape: (2741, 4)
y_train shape: (10556,)
y_test shape: (2741,)


In [19]:
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

overlap = train_groups.intersection(test_groups)

print("Unique feature groups in train:", len(train_groups))
print("Unique feature groups in test:", len(test_groups))
print("Overlapping feature groups:", len(overlap))

assert len(overlap) == 0, "Group leakage detected!"

Unique feature groups in train: 7487
Unique feature groups in test: 1872
Overlapping feature groups: 0


In [20]:
categorical_features = [
    "propertytype",
    "location"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [21]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [22]:
results = []

for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

    print(f"\n{model_name}")
    print(f"MAE: {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R²: {r2:.4f}")


Linear Regression
MAE: 6,085,440.98
RMSE: 15,397,549.56
R²: 0.4078

Decision Tree
MAE: 6,051,818.91
RMSE: 17,958,015.65
R²: 0.1944

Random Forest
MAE: 5,177,201.11
RMSE: 15,341,311.61
R²: 0.4121


In [23]:
comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by="R²",
    ascending=False
).reset_index(drop=True)

comparison_df

,Model,MAE,RMSE,R²
0,Random Forest,5.177201e+06,1.534131e+07,0.412085
1,Linear Regression,6.085441e+06,1.539755e+07,0.407767
2,Decision Tree,6.051819e+06,1.795802e+07,0.194424


In [24]:
best_model = comparison_df.iloc[0]

print("Best model based on R²:")
print(best_model["Model"])
print(f"R²: {best_model['R²']:.4f}")

Best model based on R²:
Random Forest
R²: 0.4121


# Model Comparison — Conclusion

Three regression models were evaluated using the same grouped train/test split:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

The split uses `GroupShuffleSplit` with `random_state=42` and groups based on:

- `bhk`
- `propertytype`
- `location`
- `sqft`

The train/test split was verified to contain zero overlapping feature groups.

Therefore, the reported evaluation metrics are based on a group-aware split rather than a standard random split that could allow repeated feature groups across train and test.

The models were compared using:

- MAE
- RMSE
- R²

The model with the strongest evaluation performance is selected for further investigation.